# EMS741 — Few-Shot Medical Image Segmentation
## Complete Comparison: Methods + Improvements + Combinations

### Structure
- **Section 0** — Setup (always run first)
- **Section 1** — Reptile
- **Section 2** — FOMAML
- **Section 3** — Full MAML
- **Section 4** — Meta-SGD
- **Section 5** — Transductive Fine-Tuning
- **Section 6** — iMAML (new)
- **Section 7** — Task-Weighted Reptile (new)
- **Section 8** — Baseline (UNet from scratch)
- **Section 9** — FOMAML + Focal Loss
- **Section 10** — FOMAML + Augmentation
- **Section 11** — FOMAML + Attention U-Net
- **Section 12** — FOMAML + Focal + Aug
- **Section 13** — FOMAML + Focal + Attention
- **Section 14** — FOMAML + Aug + Attention
- **Section 15** — FOMAML + All Improvements
- **Section 16** — Full Evaluation, Plots & Save

> Run **Section 0** first. Each section is independent — define, train, evaluate.
> All models saved as .pth files so you never retrain unnecessarily.


---
# Section 0 — Setup
> **Run ALL cells in this section first.**


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'torch','torchvision','matplotlib','numpy','Pillow','tqdm','-q'])
print('Packages ready.')


Packages ready.


In [3]:
import urllib.request, zipfile, os
from tqdm import tqdm
class DownloadProgress:
    def __init__(self): self.pbar = None
    def __call__(self, b, bs, ts):
        if self.pbar is None:
            self.pbar = tqdm(total=ts, unit='B', unit_scale=True, desc='Downloading')
        self.pbar.update(bs)
url = 'https://zenodo.org/records/18745413/files/ems741_cw_data.zip?download=1'
if not os.path.exists('data.zip'):
    urllib.request.urlretrieve(url, 'data.zip', DownloadProgress())
    print('Download complete!')
else:
    print('Already downloaded.')
if not os.path.exists('train'):
    with zipfile.ZipFile('data.zip','r') as z: z.extractall('./')
    print('Extracted!')
else:
    print('Already extracted.')
for split in ['train','val','test']:
    print(f'{split}: {sorted(os.listdir(split))}')


Already downloaded.
Already extracted.
train: ['task_2', 'task_3', 'task_5', 'task_7']
val: ['task_4', 'task_6']
test: ['task_1', 'task_8']


In [4]:
import os, random, copy, json, csv, pickle, shutil
import numpy as np
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [5]:
DATA_ROOT      = './'
IMAGE_SIZE     = (128, 128)
OUTER_STEPS    = 5000
INNER_STEPS    = 16
INNER_LR       = 1e-3
OUTER_LR       = 0.001
MAML_LR        = 1e-3
BATCH_SIZE     = 8
N_SHOT_LIST    = [1, 3, 5]
ADAPT_STEPS    = 100
ADAPT_LR       = 1e-3
BASELINE_STEPS = 300
BASELINE_LR    = 1e-3
IMAML_LAM      = 2.0
print('Config set.')


Config set.


In [6]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class TaskDataset(Dataset):
    def __init__(self, task_path, image_size=IMAGE_SIZE):
        self.image_size = image_size
        self.img_dir  = os.path.join(task_path,'images')
        self.mask_dir = os.path.join(task_path,'masks')
        self.filenames = sorted(os.listdir(self.img_dir))
    def __len__(self): return len(self.filenames)
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img  = Image.open(os.path.join(self.img_dir,  fname)).convert('L')
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert('L')
        img  = TF.to_tensor(TF.resize(img,  list(self.image_size)))
        mask = (TF.to_tensor(TF.resize(mask, list(self.image_size))) > 0.5).float()
        return img, mask

def get_tasks(split, root=DATA_ROOT):
    sp = os.path.join(root, split)
    return [os.path.join(sp,t) for t in sorted(os.listdir(sp))
            if os.path.isdir(os.path.join(sp,t))]

def get_few_shot_batch(task_path, n_shot):
    ds  = TaskDataset(task_path)
    idx = random.sample(range(len(ds)), k=min(n_shot, len(ds)))
    imgs, masks = zip(*[ds[i] for i in idx])
    return torch.stack(imgs).to(DEVICE), torch.stack(masks).to(DEVICE)

def augment_pair(img, mask):
    if random.random() > 0.5: img = TF.hflip(img); mask = TF.hflip(mask)
    if random.random() > 0.5: img = TF.vflip(img); mask = TF.vflip(mask)
    angle = random.uniform(-15, 15)
    img  = TF.rotate(img,  angle)
    mask = TF.rotate(mask, angle)
    img  = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
    return img, mask

def get_aug_batch(task_path, batch_size):
    ds  = TaskDataset(task_path)
    idx = random.choices(range(len(ds)), k=batch_size)
    pairs = [augment_pair(ds[i][0], ds[i][1]) for i in idx]
    imgs, masks = zip(*pairs)
    return torch.stack(imgs).to(DEVICE), torch.stack(masks).to(DEVICE)

train_tasks = get_tasks('train')
val_tasks   = get_tasks('val')
test_tasks  = get_tasks('test')
print(f'Train: {[os.path.basename(t) for t in train_tasks]}')
print(f'Val:   {[os.path.basename(t) for t in val_tasks]}')
print(f'Test:  {[os.path.basename(t) for t in test_tasks]}')


Train: ['task_2', 'task_3', 'task_5', 'task_7']
Val:   ['task_4', 'task_6']
Test:  ['task_1', 'task_8']


In [7]:
# ── Standard U-Net ───────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[16,32,64,128]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(ConvBlock(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f
        self.bottleneck = ConvBlock(features[-1], features[-1]*2)
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        ch = features[-1]*2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.decoders.append(ConvBlock(f*2, f))
            ch = f
        self.final_conv = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x); x = torch.cat([x, skip], dim=1); x = dec(x)
        return self.final_conv(x)

# ── Attention U-Net ──────────────────────────────────────────────────────────
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True),
                                  nn.GroupNorm(max(1,F_int//4), F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True),
                                  nn.GroupNorm(max(1,F_int//4), F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),
                                  nn.GroupNorm(1,1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        if g1.shape != x1.shape:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g1 + x1))

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[16,32,64,128]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(ConvBlock(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f
        self.bottleneck = ConvBlock(features[-1], features[-1]*2)
        self.upconvs    = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        ch = features[-1]*2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.attentions.append(AttentionGate(F_g=f, F_l=f, F_int=max(1,f//2)))
            self.decoders.append(ConvBlock(f*2, f))
            ch = f
        self.final_conv = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for upconv, att, dec, skip in zip(
                self.upconvs, self.attentions, self.decoders, reversed(skips)):
            x    = upconv(x)
            skip = att(x, skip)
            x    = torch.cat([x, skip], dim=1); x = dec(x)
        return self.final_conv(x)

# ── Meta-SGD wrapper ─────────────────────────────────────────────────────────
class MetaSGD(nn.Module):
    def __init__(self, base_model, init_lr=1e-3):
        super().__init__()
        self.base_model = base_model
        self.lrs = nn.ParameterList([
            nn.Parameter(torch.ones_like(p) * init_lr)
            for p in base_model.parameters()
        ])

_m = UNet().to(DEVICE); _d = torch.randn(2,1,*IMAGE_SIZE).to(DEVICE)
print(f'UNet     params: {sum(p.numel() for p in _m.parameters()):,}')
_a = AttentionUNet().to(DEVICE)
print(f'AttUNet  params: {sum(p.numel() for p in _a.parameters()):,}')
del _m, _d, _a


UNet     params: 1,942,289
AttUNet  params: 1,964,901


In [8]:
# ── Loss Functions ────────────────────────────────────────────────────────────
def dice_score(pred_logits, target, threshold=0.5, eps=1e-6):
    pred  = (torch.sigmoid(pred_logits) > threshold).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2.0*inter + eps)/(union + eps)).mean()

def dice_loss(pred_logits, target, eps=1e-6):
    pred  = torch.sigmoid(pred_logits)
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return 1.0 - ((2.0*inter + eps)/(union + eps)).mean()

def bce_dice_loss(pred_logits, target):
    return nn.BCEWithLogitsLoss()(pred_logits, target) + dice_loss(pred_logits, target)

def focal_loss(pred_logits, target, alpha=0.8, gamma=2.0):
    bce    = nn.BCEWithLogitsLoss(reduction='none')(pred_logits, target)
    probs  = torch.sigmoid(pred_logits)
    p_t    = probs * target + (1 - probs) * (1 - target)
    alpha_t = alpha * target + (1 - alpha) * (1 - target)
    return (alpha_t * (1 - p_t) ** gamma * bce).mean()

def focal_dice_loss(pred_logits, target):
    return focal_loss(pred_logits, target) + dice_loss(pred_logits, target)

combined_loss = bce_dice_loss
print('Loss functions defined.')


Loss functions defined.


In [9]:
# ── Shared Evaluation Helpers ─────────────────────────────────────────────────
def few_shot_adapt(meta_model, task_path, n_shot,
                    adapt_steps=ADAPT_STEPS, adapt_lr=ADAPT_LR,
                    use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    adapted   = copy.deepcopy(meta_model); adapted.train()
    optimizer = optim.Adam(adapted.parameters(), lr=adapt_lr)
    s_imgs, s_masks = get_few_shot_batch(task_path, n_shot)
    for _ in range(adapt_steps):
        if use_augmentation:
            pairs = [augment_pair(s_imgs[i].cpu(), s_masks[i].cpu()) for i in range(len(s_imgs))]
            xi = torch.stack([p[0] for p in pairs]).to(DEVICE)
            xm = torch.stack([p[1] for p in pairs]).to(DEVICE)
        else:
            xi, xm = s_imgs, s_masks
        loss = loss_fn(adapted(xi), xm)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    adapted.eval()
    return adapted

def evaluate_on_task(model, task_path, n_shot, adapt_steps=ADAPT_STEPS,
                      use_augmentation=False, loss_fn=None):
    adapted = few_shot_adapt(model, task_path, n_shot, adapt_steps,
                              use_augmentation=use_augmentation, loss_fn=loss_fn)
    loader  = DataLoader(TaskDataset(task_path), batch_size=8, shuffle=False)
    dices   = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            dices.append(dice_score(adapted(imgs), masks).item())
    return float(np.mean(dices))

def quick_evaluate(model, task_paths, n_shot, adapt_steps=ADAPT_STEPS,
                    use_augmentation=False, loss_fn=None):
    return float(np.mean([
        evaluate_on_task(model, tp, n_shot, adapt_steps, use_augmentation, loss_fn)
        for tp in task_paths
    ]))

def metasgd_evaluate_on_task(model, task_path, n_shot,
                               adapt_steps=ADAPT_STEPS, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    adapted   = copy.deepcopy(model.base_model); adapted.train()
    mean_lr   = float(torch.stack([lr.abs().mean() for lr in model.lrs]).mean())
    optimizer = optim.Adam(adapted.parameters(), lr=mean_lr)
    s_imgs, s_masks = get_few_shot_batch(task_path, n_shot)
    for _ in range(adapt_steps):
        loss = loss_fn(adapted(s_imgs), s_masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    adapted.eval()
    loader = DataLoader(TaskDataset(task_path), batch_size=8, shuffle=False)
    dices  = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            dices.append(dice_score(adapted(imgs), masks).item())
    return float(np.mean(dices))

def transductive_adapt(meta_model, task_path, n_shot,
                        adapt_steps=ADAPT_STEPS, adapt_lr=ADAPT_LR,
                        entropy_weight=0.1, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    def _ent(logits):
        p = torch.sigmoid(logits)
        return -(p*torch.log(p+1e-8)+(1-p)*torch.log(1-p+1e-8)).mean()
    adapted   = copy.deepcopy(meta_model); adapted.train()
    optimizer = optim.Adam(adapted.parameters(), lr=adapt_lr)
    s_imgs, s_masks = get_few_shot_batch(task_path, n_shot)
    dataset    = TaskDataset(task_path)
    query_imgs = torch.stack([dataset[i][0] for i in range(len(dataset))]).to(DEVICE)
    for _ in range(adapt_steps):
        sup  = loss_fn(adapted(s_imgs), s_masks)
        ent  = _ent(adapted(query_imgs[random.choices(range(len(query_imgs)),k=8)]))
        loss = sup + entropy_weight * ent
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    adapted.eval()
    return adapted

def transductive_evaluate_on_task(meta_model, task_path, n_shot, loss_fn=None):
    adapted = transductive_adapt(meta_model, task_path, n_shot, loss_fn=loss_fn)
    loader  = DataLoader(TaskDataset(task_path), batch_size=8, shuffle=False)
    dices   = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            dices.append(dice_score(adapted(imgs), masks).item())
    return float(np.mean(dices))

# ── Shared results dict ───────────────────────────────────────────────────────
results = {}
losses_dict = {}
print('All helpers defined. Setup complete!')


All helpers defined. Setup complete!


---
# Shared Training Functions
> **Run this once after Section 0.**


In [10]:
# ── Reptile ──────────────────────────────────────────────────────────────────
def reptile_train(model, train_tasks, outer_steps=OUTER_STEPS,
                   inner_steps=INNER_STEPS, inner_lr=INNER_LR,
                   outer_lr=OUTER_LR, batch_size=BATCH_SIZE,
                   val_tasks=None, val_interval=200,
                   use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    model.train()
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='Reptile'):
        weights_old = {k: v.clone() for k, v in model.state_dict().items()}
        task_path   = random.choice(train_tasks)
        task_ds     = TaskDataset(task_path)
        inner_opt   = optim.SGD(model.parameters(), lr=inner_lr)
        step_loss, valid_steps = 0.0, 0
        for _ in range(inner_steps):
            x, y = get_aug_batch(task_path, batch_size) if use_augmentation else (
                lambda idx: (torch.stack([task_ds[i][0] for i in idx]).to(DEVICE),
                             torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)))\
                    (random.choices(range(len(task_ds)), k=batch_size))
            loss = loss_fn(model(x), y)
            if torch.isnan(loss): continue
            inner_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            inner_opt.step()
            step_loss += loss.item(); valid_steps += 1
        train_losses.append(step_loss / max(valid_steps, 1))
        wn = model.state_dict()
        if all(not torch.isnan(wn[k].float()).any() for k in weights_old):
            model.load_state_dict({k: weights_old[k] + outer_lr*(wn[k].float()-weights_old[k].float())
                                   for k in weights_old})
        else:
            model.load_state_dict(weights_old)
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {train_losses[-1]:.4f} | val: {v:.4f}')
    return train_losses, val_dices

# ── FOMAML ───────────────────────────────────────────────────────────────────
def fomaml_train(model, train_tasks, outer_steps=OUTER_STEPS,
                  inner_steps=INNER_STEPS, inner_lr=INNER_LR,
                  outer_lr=MAML_LR, batch_size=BATCH_SIZE,
                  val_tasks=None, val_interval=200,
                  use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    outer_opt = optim.Adam(model.parameters(), lr=outer_lr)
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='FOMAML'):
        temp      = copy.deepcopy(model); temp.train()
        inner_opt = optim.SGD(temp.parameters(), lr=inner_lr)
        task_path = random.choice(train_tasks)
        task_ds   = TaskDataset(task_path)
        for _ in range(inner_steps):
            if use_augmentation:
                x, y = get_aug_batch(task_path, batch_size)
            else:
                idx = random.choices(range(len(task_ds)), k=batch_size)
                x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
                y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
            loss = loss_fn(temp(x), y)
            inner_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(temp.parameters(), 1.0)
            inner_opt.step()
        idx = random.choices(range(len(task_ds)), k=batch_size)
        x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
        y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
        q_loss = loss_fn(temp(x), y)
        train_losses.append(q_loss.item())
        q_loss.backward()
        outer_opt.zero_grad()
        for mp, tp in zip(model.parameters(), temp.parameters()):
            if tp.grad is not None: mp.grad = tp.grad.clone()
        outer_opt.step()
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {train_losses[-1]:.4f} | val: {v:.4f}')
    return train_losses, val_dices

# ── Full MAML ────────────────────────────────────────────────────────────────
def functional_forward(model, fast_weights, x, y, loss_fn):
    backup = [p.data.clone() for p in model.parameters()]
    for p, fw in zip(model.parameters(), fast_weights): p.data = fw.data
    loss = loss_fn(model(x), y)
    for p, b in zip(model.parameters(), backup): p.data = b
    return loss

def maml_train(model, train_tasks, outer_steps=OUTER_STEPS,
               inner_steps=INNER_STEPS, inner_lr=INNER_LR,
               outer_lr=MAML_LR, batch_size=BATCH_SIZE,
               val_tasks=None, val_interval=200,
               use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    outer_opt = optim.Adam(model.parameters(), lr=outer_lr)
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='Full MAML'):
        task_path = random.choice(train_tasks)
        task_ds   = TaskDataset(task_path)
        fast_w    = [p.clone() for p in model.parameters()]
        for _ in range(inner_steps):
            if use_augmentation:
                x, y = get_aug_batch(task_path, batch_size)
            else:
                idx = random.choices(range(len(task_ds)), k=batch_size)
                x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
                y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
            loss  = functional_forward(model, fast_w, x, y, loss_fn)
            grads = torch.autograd.grad(loss, fast_w, create_graph=True, allow_unused=True)
            fast_w = [w - inner_lr*(g if g is not None else torch.zeros_like(w))
                      for w, g in zip(fast_w, grads)]
        idx = random.choices(range(len(task_ds)), k=batch_size)
        x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
        y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
        q_loss = functional_forward(model, fast_w, x, y, loss_fn)
        train_losses.append(q_loss.item())
        outer_opt.zero_grad(); q_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        outer_opt.step()
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {train_losses[-1]:.4f} | val: {v:.4f}')
    return train_losses, val_dices

# ── Meta-SGD ─────────────────────────────────────────────────────────────────
def metasgd_train(model, train_tasks, outer_steps=OUTER_STEPS,
                   inner_steps=INNER_STEPS, outer_lr=MAML_LR,
                   batch_size=BATCH_SIZE, val_tasks=None, val_interval=200,
                   use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    outer_opt = optim.Adam(model.parameters(), lr=outer_lr)
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='Meta-SGD'):
        temp     = copy.deepcopy(model); temp.train()
        task_path = random.choice(train_tasks)
        task_ds  = TaskDataset(task_path)
        fast_w   = list(temp.base_model.parameters())
        fast_lrs = list(temp.lrs)
        for _ in range(inner_steps):
            if use_augmentation:
                x, y = get_aug_batch(task_path, batch_size)
            else:
                idx = random.choices(range(len(task_ds)), k=batch_size)
                x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
                y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
            loss  = loss_fn(temp.base_model(x), y)
            grads = torch.autograd.grad(loss, fast_w, create_graph=False, allow_unused=True)
            fast_w = [w - lr.abs()*(g if g is not None else torch.zeros_like(w))
                      for w, lr, g in zip(fast_w, fast_lrs, grads)]
            for p, fw in zip(temp.base_model.parameters(), fast_w): p.data = fw.data
        idx = random.choices(range(len(task_ds)), k=batch_size)
        x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
        y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
        q_loss = loss_fn(temp.base_model(x), y)
        train_losses.append(q_loss.item())
        outer_opt.zero_grad(); q_loss.backward()
        for mp, tp in zip(model.base_model.parameters(), temp.base_model.parameters()):
            if tp.grad is not None: mp.grad = tp.grad.clone()
        for ml, tl in zip(model.lrs, temp.lrs):
            if tl.grad is not None: ml.grad = tl.grad.clone()
        outer_opt.step()
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model.base_model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {train_losses[-1]:.4f} | val: {v:.4f}')
    return train_losses, val_dices

# ── iMAML ────────────────────────────────────────────────────────────────────
def imaml_train(model, train_tasks, outer_steps=OUTER_STEPS,
                 inner_steps=INNER_STEPS, inner_lr=INNER_LR,
                 outer_lr=MAML_LR, batch_size=BATCH_SIZE,
                 lam=IMAML_LAM, val_tasks=None, val_interval=200,
                 use_augmentation=False, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    outer_opt = optim.Adam(model.parameters(), lr=outer_lr)
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='iMAML'):
        task_path = random.choice(train_tasks)
        task_ds   = TaskDataset(task_path)
        theta_0   = [p.clone().detach() for p in model.parameters()]
        adapted   = copy.deepcopy(model); adapted.train()
        inner_opt = optim.SGD(adapted.parameters(), lr=inner_lr)
        for _ in range(inner_steps):
            if use_augmentation:
                x, y = get_aug_batch(task_path, batch_size)
            else:
                idx = random.choices(range(len(task_ds)), k=batch_size)
                x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
                y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
            task_loss = loss_fn(adapted(x), y)
            # Proximal term: pulls towards original weights
            prox = sum(((p - p0)**2).sum()
                       for p, p0 in zip(adapted.parameters(), theta_0))
            total_loss = task_loss + (lam / 2.0) * prox
            inner_opt.zero_grad(); total_loss.backward(); inner_opt.step()
        # Query loss for outer update
        idx = random.choices(range(len(task_ds)), k=batch_size)
        x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
        y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
        q_loss = loss_fn(adapted(x), y)
        train_losses.append(q_loss.item())
        outer_opt.zero_grad(); q_loss.backward()
        for mp, ap in zip(model.parameters(), adapted.parameters()):
            if ap.grad is not None: mp.grad = ap.grad.clone()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        outer_opt.step()
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {train_losses[-1]:.4f} | val: {v:.4f}')
    return train_losses, val_dices

# ── Task-Weighted Reptile ────────────────────────────────────────────────────
def weighted_reptile_train(model, train_tasks, outer_steps=OUTER_STEPS,
                            inner_steps=INNER_STEPS, inner_lr=INNER_LR,
                            outer_lr=OUTER_LR, batch_size=BATCH_SIZE,
                            val_tasks=None, val_interval=200,
                            use_augmentation=False, loss_fn=None):
    """
    Reptile with importance-weighted outer update.
    Tasks with lower loss (easier) get lower weight — focuses learning on harder tasks.
    """
    if loss_fn is None: loss_fn = bce_dice_loss
    model.train()
    train_losses, val_dices = [], []
    for outer_step in tqdm(range(outer_steps), desc='Weighted Reptile'):
        weights_old = {k: v.clone() for k, v in model.state_dict().items()}
        task_path   = random.choice(train_tasks)
        task_ds     = TaskDataset(task_path)
        inner_opt   = optim.SGD(model.parameters(), lr=inner_lr)
        step_loss, valid_steps = 0.0, 0
        for _ in range(inner_steps):
            if use_augmentation:
                x, y = get_aug_batch(task_path, batch_size)
            else:
                idx = random.choices(range(len(task_ds)), k=batch_size)
                x = torch.stack([task_ds[i][0] for i in idx]).to(DEVICE)
                y = torch.stack([task_ds[i][1] for i in idx]).to(DEVICE)
            loss = loss_fn(model(x), y)
            if torch.isnan(loss): continue
            inner_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            inner_opt.step()
            step_loss += loss.item(); valid_steps += 1
        avg_loss = step_loss / max(valid_steps, 1)
        train_losses.append(avg_loss)
        # Weight by inverse loss: harder tasks get more weight
        task_weight = 1.0 / (avg_loss + 1e-6)
        task_weight = min(task_weight, 5.0)  # clip to avoid extreme weights
        effective_lr = outer_lr * task_weight / 5.0
        wn = model.state_dict()
        if all(not torch.isnan(wn[k].float()).any() for k in weights_old):
            model.load_state_dict(
                {k: weights_old[k] + effective_lr*(wn[k].float()-weights_old[k].float())
                 for k in weights_old})
        else:
            model.load_state_dict(weights_old)
        if val_tasks and (outer_step+1) % val_interval == 0:
            v = quick_evaluate(model, val_tasks, 5, loss_fn=loss_fn)
            val_dices.append((outer_step+1, v))
            tqdm.write(f'Step {outer_step+1:4d} | loss: {avg_loss:.4f} | val: {v:.4f}')
    return train_losses, val_dices

print('All training functions defined!')


All training functions defined!


---
# Section 1 — Reptile


In [20]:
reptile_model = UNet().to(DEVICE)
reptile_losses, _ = reptile_train(reptile_model, train_tasks, val_tasks=val_tasks)
torch.save(reptile_model.state_dict(), 'reptile.pth')
losses_dict['Reptile'] = reptile_losses
print('Reptile saved.')


Reptile:   4%|██▋                                                                 | 200/5000 [02:42<5:32:59,  4.16s/it]

Step  200 | loss: 1.9981 | val: 0.3379


Reptile:   8%|█████▍                                                              | 400/5000 [05:25<4:28:53,  3.51s/it]

Step  400 | loss: 2.0071 | val: 0.3350


Reptile:  12%|████████▏                                                           | 600/5000 [08:09<4:32:14,  3.71s/it]

...

Step 4600 | loss: 1.8092 | val: 0.2818


Reptile:  96%|████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:08:33<12:33,  3.77s/it]

Step 4800 | loss: 1.8347 | val: 0.2374


Reptile: 100%|███████████████████████████████████████████████████████████████████| 5000/5000 [1:11:35<00:00,  1.16it/s]

Step 5000 | loss: 1.8088 | val: 0.2310
Reptile saved.


In [21]:
reptile_model = UNet().to(DEVICE)
reptile_model.load_state_dict(torch.load('reptile.pth', map_location=DEVICE))
results['Reptile'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(reptile_model, tp, n) for tp in test_tasks]
    results['Reptile'][n] = d
    print(f'Reptile {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


Reptile 1-shot: 0.2462 +/- 0.0703
Reptile 3-shot: 0.0801 +/- 0.0329
Reptile 5-shot: 0.2811 +/- 0.1909


---
# Section 2 — FOMAML


In [22]:
fomaml_model = UNet().to(DEVICE)
fomaml_losses, _ = fomaml_train(fomaml_model, train_tasks, val_tasks=val_tasks)
torch.save(fomaml_model.state_dict(), 'fomaml.pth')
losses_dict['FOMAML'] = fomaml_losses
print('FOMAML saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [02:34<4:58:01,  3.73s/it]

Step  200 | loss: 1.1433 | val: 0.2696


FOMAML:   8%|█████▌                                                               | 400/5000 [05:14<5:04:26,  3.97s/it]

Step  400 | loss: 1.0534 | val: 0.3680


FOMAML:  12%|████████▎                                                            | 600/5000 [07:51<4:33:56,  3.74s/it]

...

Step 4600 | loss: 0.7561 | val: 0.4502


FOMAML:  96%|█████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:01:55<11:50,  3.55s/it]

Step 4800 | loss: 0.4204 | val: 0.4267


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [1:04:24<00:00,  1.29it/s]

Step 5000 | loss: 0.8163 | val: 0.3753
FOMAML saved.


In [23]:
fomaml_model = UNet().to(DEVICE)
fomaml_model.load_state_dict(torch.load('fomaml.pth', map_location=DEVICE))
results['FOMAML'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_model, tp, n) for tp in test_tasks]
    results['FOMAML'][n] = d
    print(f'FOMAML {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


FOMAML 1-shot: 0.0415 +/- 0.0192
FOMAML 3-shot: 0.3146 +/- 0.1712
FOMAML 5-shot: 0.3434 +/- 0.1983


---
# Section 3 — Full MAML


In [24]:
maml_model = UNet().to(DEVICE)
maml_losses, _ = maml_train(maml_model, train_tasks, val_tasks=val_tasks)
torch.save(maml_model.state_dict(), 'maml.pth')
losses_dict['FullMAML'] = maml_losses
print('Full MAML saved.')


Full MAML:   4%|██▋                                                               | 200/5000 [02:13<4:17:56,  3.22s/it]

Step  200 | loss: 1.1116 | val: 0.4047


Full MAML:   8%|█████▎                                                            | 400/5000 [04:23<4:04:31,  3.19s/it]

Step  400 | loss: 1.0805 | val: 0.3869


Full MAML:  12%|███████▉                                                          | 600/5000 [06:32<3:52:21,  3.17s/it]

...

Step 4600 | loss: 0.7550 | val: 0.4451


Full MAML:  96%|████████████████████████████████████████████████████████████████▎  | 4800/5000 [53:11<11:30,  3.45s/it]

Step 4800 | loss: 0.9112 | val: 0.3513


Full MAML: 100%|███████████████████████████████████████████████████████████████████| 5000/5000 [55:26<00:00,  1.50it/s]

Step 5000 | loss: 0.6437 | val: 0.3698
Full MAML saved.


In [25]:
maml_model = UNet().to(DEVICE)
maml_model.load_state_dict(torch.load('maml.pth', map_location=DEVICE))
results['FullMAML'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(maml_model, tp, n) for tp in test_tasks]
    results['FullMAML'][n] = d
    print(f'Full MAML {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


Full MAML 1-shot: 0.2423 +/- 0.1831
Full MAML 3-shot: 0.0568 +/- 0.0035
Full MAML 5-shot: 0.3246 +/- 0.1779


---
# Section 4 — Meta-SGD


In [11]:
metasgd_model = MetaSGD(UNet().to(DEVICE)).to(DEVICE)
metasgd_losses, _ = metasgd_train(metasgd_model, train_tasks, val_tasks=val_tasks)
torch.save(metasgd_model.state_dict(), 'metasgd.pth')
losses_dict['MetaSGD'] = metasgd_losses
print('Meta-SGD saved.')


Meta-SGD:   4%|██▋                                                               | 200/5000 [10:07<27:49:06, 20.86s/it]

Step  200 | loss: 1.1376 | val: 0.3041


Meta-SGD:   8%|█████▎                                                            | 400/5000 [18:14<21:20:23, 16.70s/it]

Step  400 | loss: 1.0391 | val: 0.3486


Meta-SGD:  12%|███████▉                                                          | 600/5000 [26:13<20:06:08, 16.45s/it]

...

Step 4600 | loss: 0.5840 | val: 0.2075


Meta-SGD:  96%|███████████████████████████████████████████████████████████████▎  | 4800/5000 [3:06:40<53:56, 16.18s/it]

Step 4800 | loss: 0.7843 | val: 0.2055


Meta-SGD: 100%|██████████████████████████████████████████████████████████████████| 5000/5000 [3:14:08<00:00,  2.33s/it]

Step 5000 | loss: 0.5515 | val: 0.2987
Meta-SGD saved.


In [12]:
metasgd_model = MetaSGD(UNet().to(DEVICE)).to(DEVICE)
metasgd_model.load_state_dict(torch.load('metasgd.pth', map_location=DEVICE))
results['MetaSGD'] = {}
for n in N_SHOT_LIST:
    d = [metasgd_evaluate_on_task(metasgd_model, tp, n) for tp in test_tasks]
    results['MetaSGD'][n] = d
    print(f'Meta-SGD {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


C:\Users\hamad\AppData\Local\Temp\ipykernel_48640\3706123672.py:44: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  mean_lr   = float(torch.stack([lr.abs().mean() for lr in model.lrs]).mean())


Meta-SGD 1-shot: 0.1899 +/- 0.0229
Meta-SGD 3-shot: 0.3209 +/- 0.1663
Meta-SGD 5-shot: 0.2924 +/- 0.0866


---
# Section 5 — Transductive Fine-Tuning
> No training — uses fomaml.pth as base. Run Section 2 first.


In [13]:
try:
    fomaml_model
except NameError:
    fomaml_model = UNet().to(DEVICE)
    fomaml_model.load_state_dict(torch.load('fomaml.pth', map_location=DEVICE))
    print('Loaded fomaml from checkpoint.')

results['Transductive'] = {}
for n in N_SHOT_LIST:
    d = [transductive_evaluate_on_task(fomaml_model, tp, n) for tp in test_tasks]
    results['Transductive'][n] = d
    print(f'Transductive {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')
print('Transductive done.')


Loaded fomaml from checkpoint.
Transductive 1-shot: 0.2170 +/- 0.1688
Transductive 3-shot: 0.3180 +/- 0.1740
Transductive 5-shot: 0.3621 +/- 0.2304
Transductive done.


---
# Section 6 — iMAML
> Implicit MAML — uses proximal regularisation instead of second-order gradients. More stable than Full MAML.


In [14]:
imaml_model = UNet().to(DEVICE)
imaml_losses, _ = imaml_train(imaml_model, train_tasks, val_tasks=val_tasks)
torch.save(imaml_model.state_dict(), 'imaml.pth')
losses_dict['iMAML'] = imaml_losses
print('iMAML saved.')


iMAML:   4%|██▊                                                                  | 200/5000 [14:20<24:10:25, 18.13s/it]

Step  200 | loss: 1.0361 | val: 0.0953


iMAML:   8%|█████▌                                                               | 400/5000 [28:40<22:51:06, 17.88s/it]

Step  400 | loss: 1.1951 | val: 0.0953


iMAML:  12%|████████▎                                                            | 600/5000 [42:49<23:02:56, 18.86s/it]

...

Step 4600 | loss: 0.3200 | val: 0.3967


iMAML:  96%|██████████████████████████████████████████████████████████████████▏  | 4800/5000 [5:42:12<59:51, 17.96s/it]

Step 4800 | loss: 0.4436 | val: 0.3948


iMAML: 100%|█████████████████████████████████████████████████████████████████████| 5000/5000 [5:56:38<00:00,  4.28s/it]

Step 5000 | loss: 0.1891 | val: 0.3532
iMAML saved.


In [15]:
imaml_model = UNet().to(DEVICE)
imaml_model.load_state_dict(torch.load('imaml.pth', map_location=DEVICE))
results['iMAML'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(imaml_model, tp, n) for tp in test_tasks]
    results['iMAML'][n] = d
    print(f'iMAML {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


iMAML 1-shot: 0.2638 +/- 0.0400
iMAML 3-shot: 0.3990 +/- 0.1571
iMAML 5-shot: 0.3372 +/- 0.1499


---
# Section 7 — Task-Weighted Reptile
> Reptile with importance weighting: harder tasks get larger outer update.


In [16]:
wreptile_model = UNet().to(DEVICE)
wreptile_losses, _ = weighted_reptile_train(wreptile_model, train_tasks, val_tasks=val_tasks)
torch.save(wreptile_model.state_dict(), 'wreptile.pth')
losses_dict['WeightedReptile'] = wreptile_losses
print('Weighted Reptile saved.')


Weighted Reptile:   4%|██▎                                                       | 200/5000 [12:45<23:04:12, 17.30s/it]

Step  200 | loss: 1.5509 | val: 0.3691


Weighted Reptile:   8%|████▋                                                     | 400/5000 [25:35<22:41:45, 17.76s/it]

Step  400 | loss: 1.5628 | val: 0.3330


Weighted Reptile:  12%|██████▉                                                   | 600/5000 [38:23<21:27:57, 17.56s/it]

...

Step 4600 | loss: 1.5451 | val: 0.3244


Weighted Reptile:  96%|█████████████████████████████████████████████████████▊  | 4800/5000 [5:15:15<1:07:31, 20.26s/it]

Step 4800 | loss: 1.5437 | val: 0.2709


Weighted Reptile: 100%|██████████████████████████████████████████████████████████| 5000/5000 [5:29:59<00:00,  3.96s/it]

Step 5000 | loss: 1.5416 | val: 0.4347
Weighted Reptile saved.


In [17]:
wreptile_model = UNet().to(DEVICE)
wreptile_model.load_state_dict(torch.load('wreptile.pth', map_location=DEVICE))
results['WeightedReptile'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(wreptile_model, tp, n) for tp in test_tasks]
    results['WeightedReptile'][n] = d
    print(f'Weighted Reptile {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')


Weighted Reptile 1-shot: 0.1832 +/- 0.0969
Weighted Reptile 3-shot: 0.1952 +/- 0.0107
Weighted Reptile 5-shot: 0.3883 +/- 0.1645


---
# Section 8 — Baseline
> Fresh UNet trained from scratch on n-shot examples only. No meta-learning.


In [18]:
def baseline_eval(task_path, n_shot, train_steps=BASELINE_STEPS, lr=BASELINE_LR, loss_fn=None):
    if loss_fn is None: loss_fn = bce_dice_loss
    model = UNet().to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr)
    s_imgs, s_masks = get_few_shot_batch(task_path, n_shot)
    model.train()
    for _ in range(train_steps):
        loss = loss_fn(model(s_imgs), s_masks)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    loader = DataLoader(TaskDataset(task_path), batch_size=8, shuffle=False)
    dices  = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            dices.append(dice_score(model(imgs), masks).item())
    return float(np.mean(dices))

results['Baseline'] = {}
for n in N_SHOT_LIST:
    d = [baseline_eval(tp, n) for tp in test_tasks]
    results['Baseline'][n] = d
    print(f'Baseline {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}')
print('Baseline done.')


Baseline 1-shot: 0.2889 +/- 0.0805
Baseline 3-shot: 0.2828 +/- 0.1910
Baseline 5-shot: 0.3645 +/- 0.1407
Baseline done.


---
# Section 9 — FOMAML + Focal Loss
> Same as FOMAML but replaces BCE+Dice with Focal+Dice to fix class imbalance.


In [11]:
fomaml_focal = UNet().to(DEVICE)
fomaml_focal_losses, _ = fomaml_train(
    fomaml_focal, train_tasks, val_tasks=val_tasks, loss_fn=focal_dice_loss)
torch.save(fomaml_focal.state_dict(), 'fomaml_focal.pth')
losses_dict['FOMAML+Focal'] = fomaml_focal_losses
print('FOMAML+Focal saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [02:40<5:45:21,  4.32s/it]

Step  200 | loss: 0.8831 | val: 0.3928


FOMAML:   8%|█████▌                                                               | 400/5000 [05:22<4:56:45,  3.87s/it]

Step  400 | loss: 1.0693 | val: 0.4683


FOMAML:  12%|████████▎                                                            | 600/5000 [08:04<4:38:42,  3.80s/it]

...

Step 4600 | loss: 0.2650 | val: 0.1973


FOMAML:  96%|█████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:48:19<12:30,  3.75s/it]

Step 4800 | loss: 0.5520 | val: 0.3222


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [1:50:59<00:00,  1.33s/it]

Step 5000 | loss: 0.3620 | val: 0.4120
FOMAML+Focal saved.


In [12]:
fomaml_focal = UNet().to(DEVICE)
fomaml_focal.load_state_dict(torch.load('fomaml_focal.pth', map_location=DEVICE))
results['FOMAML+Focal'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_focal, tp, n, loss_fn=focal_dice_loss) for tp in test_tasks]
    results['FOMAML+Focal'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Focal {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


FOMAML+Focal 1-shot: 0.1508 +/- 0.0011  (vs FOMAML: +0.1508)
FOMAML+Focal 3-shot: 0.2945 +/- 0.1397  (vs FOMAML: +0.2945)
FOMAML+Focal 5-shot: 0.2767 +/- 0.1663  (vs FOMAML: +0.2767)


---
# Section 10 — FOMAML + Augmentation
> Adds random flips, rotations, brightness jitter during meta-training.


In [13]:
fomaml_aug = UNet().to(DEVICE)
fomaml_aug_losses, _ = fomaml_train(
    fomaml_aug, train_tasks, val_tasks=val_tasks, use_augmentation=True)
torch.save(fomaml_aug.state_dict(), 'fomaml_aug.pth')
losses_dict['FOMAML+Aug'] = fomaml_aug_losses
print('FOMAML+Aug saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [03:28<5:19:02,  3.99s/it]

Step  200 | loss: 1.1907 | val: 0.4126


FOMAML:   8%|█████▌                                                               | 400/5000 [06:59<5:06:41,  4.00s/it]

Step  400 | loss: 0.9577 | val: 0.4070


FOMAML:  12%|████████▎                                                            | 600/5000 [10:28<4:54:34,  4.02s/it]

...

Step 4600 | loss: 0.2570 | val: 0.4393


FOMAML:  96%|███████████████████████████████████████████████████████████████▎  | 4800/5000 [2:02:10<1:22:04, 24.62s/it]

Step 4800 | loss: 0.6228 | val: 0.3335


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [2:18:38<00:00,  1.66s/it]

Step 5000 | loss: 0.4346 | val: 0.4510
FOMAML+Aug saved.


In [14]:
fomaml_aug = UNet().to(DEVICE)
fomaml_aug.load_state_dict(torch.load('fomaml_aug.pth', map_location=DEVICE))
results['FOMAML+Aug'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_aug, tp, n, use_augmentation=True) for tp in test_tasks]
    results['FOMAML+Aug'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Aug {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


FOMAML+Aug 1-shot: 0.3316 +/- 0.1078  (vs FOMAML: +0.3316)
FOMAML+Aug 3-shot: 0.3235 +/- 0.1730  (vs FOMAML: +0.3235)
FOMAML+Aug 5-shot: 0.3907 +/- 0.2155  (vs FOMAML: +0.3907)


---
# Section 11 — FOMAML + Attention U-Net
> Replaces standard U-Net with Attention U-Net. Attention gates focus on relevant regions.


In [16]:
fomaml_att = AttentionUNet().to(DEVICE)
fomaml_att_losses, _ = fomaml_train(
    fomaml_att, train_tasks, val_tasks=val_tasks)
torch.save(fomaml_att.state_dict(), 'fomaml_att.pth')
losses_dict['FOMAML+Att'] = fomaml_att_losses
print('FOMAML+Attention saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [03:03<6:26:20,  4.83s/it]

Step  200 | loss: 1.0595 | val: 0.4853


FOMAML:   8%|█████▌                                                               | 400/5000 [06:19<6:24:55,  5.02s/it]

Step  400 | loss: 1.1791 | val: 0.4031


FOMAML:  12%|████████▎                                                            | 600/5000 [09:30<5:12:29,  4.26s/it]

...

Step 4600 | loss: 0.6850 | val: 0.3038


FOMAML:  96%|█████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:13:37<13:37,  4.09s/it]

Step 4800 | loss: 0.3657 | val: 0.3061


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [1:16:34<00:00,  1.09it/s]

Step 5000 | loss: 1.0881 | val: 0.3980
FOMAML+Attention saved.


In [17]:
fomaml_att = AttentionUNet().to(DEVICE)
fomaml_att.load_state_dict(torch.load('fomaml_att.pth', map_location=DEVICE))
results['FOMAML+Att'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_att, tp, n) for tp in test_tasks]
    results['FOMAML+Att'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Att {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


FOMAML+Att 1-shot: 0.1024 +/- 0.0192  (vs FOMAML: +0.1024)
FOMAML+Att 3-shot: 0.3623 +/- 0.1817  (vs FOMAML: +0.3623)
FOMAML+Att 5-shot: 0.3749 +/- 0.1456  (vs FOMAML: +0.3749)


---
# Section 12 — FOMAML + Focal + Aug
> Combination: Focal Loss + Augmentation.


In [18]:
fomaml_focal_aug = UNet().to(DEVICE)
fomaml_focal_aug_losses, _ = fomaml_train(
    fomaml_focal_aug, train_tasks, val_tasks=val_tasks,
    use_augmentation=True, loss_fn=focal_dice_loss)
torch.save(fomaml_focal_aug.state_dict(), 'fomaml_focal_aug.pth')
losses_dict['FOMAML+Focal+Aug'] = fomaml_focal_aug_losses
print('FOMAML+Focal+Aug saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [03:15<5:03:53,  3.80s/it]

Step  200 | loss: 0.9476 | val: 0.3008


FOMAML:   8%|█████▌                                                               | 400/5000 [06:32<4:53:31,  3.83s/it]

Step  400 | loss: 0.8558 | val: 0.3031


FOMAML:  12%|████████▎                                                            | 600/5000 [09:48<4:39:44,  3.81s/it]

...

Step 4600 | loss: 0.3860 | val: 0.3813


FOMAML:  96%|█████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:18:42<12:44,  3.82s/it]

Step 4800 | loss: 0.3742 | val: 0.4044


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [1:21:58<00:00,  1.02it/s]

Step 5000 | loss: 0.9209 | val: 0.4033
FOMAML+Focal+Aug saved.


In [19]:
fomaml_focal_aug = UNet().to(DEVICE)
fomaml_focal_aug.load_state_dict(torch.load('fomaml_focal_aug.pth', map_location=DEVICE))
results['FOMAML+Focal+Aug'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_focal_aug, tp, n,
                           use_augmentation=True, loss_fn=focal_dice_loss)
         for tp in test_tasks]
    results['FOMAML+Focal+Aug'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Focal+Aug {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


FOMAML+Focal+Aug 1-shot: 0.2828 +/- 0.0590  (vs FOMAML: +0.2828)
FOMAML+Focal+Aug 3-shot: 0.3051 +/- 0.1973  (vs FOMAML: +0.3051)
FOMAML+Focal+Aug 5-shot: 0.3366 +/- 0.1245  (vs FOMAML: +0.3366)


---
# Section 13 — FOMAML + Focal + Attention
> Combination: Focal Loss + Attention U-Net.


In [20]:
fomaml_focal_att = AttentionUNet().to(DEVICE)
fomaml_focal_att_losses, _ = fomaml_train(
    fomaml_focal_att, train_tasks, val_tasks=val_tasks, loss_fn=focal_dice_loss)
torch.save(fomaml_focal_att.state_dict(), 'fomaml_focal_att.pth')
losses_dict['FOMAML+Focal+Att'] = fomaml_focal_att_losses
print('FOMAML+Focal+Att saved.')


FOMAML:   4%|██▊                                                                  | 200/5000 [02:58<5:29:57,  4.12s/it]

Step  200 | loss: 0.9007 | val: 0.4542


FOMAML:   8%|█████▌                                                               | 400/5000 [05:57<5:15:50,  4.12s/it]

Step  400 | loss: 0.8323 | val: 0.3675


FOMAML:  12%|████████▎                                                            | 600/5000 [08:56<5:01:17,  4.11s/it]

...

Step 4600 | loss: 0.3853 | val: 0.4445


FOMAML:  96%|█████████████████████████████████████████████████████████████████▎  | 4800/5000 [1:11:30<13:44,  4.12s/it]

Step 4800 | loss: 0.2766 | val: 0.3927


FOMAML: 100%|████████████████████████████████████████████████████████████████████| 5000/5000 [1:14:29<00:00,  1.12it/s]

Step 5000 | loss: 0.5426 | val: 0.4008
FOMAML+Focal+Att saved.


In [21]:
fomaml_focal_att = AttentionUNet().to(DEVICE)
fomaml_focal_att.load_state_dict(torch.load('fomaml_focal_att.pth', map_location=DEVICE))
results['FOMAML+Focal+Att'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_focal_att, tp, n, loss_fn=focal_dice_loss)
         for tp in test_tasks]
    results['FOMAML+Focal+Att'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Focal+Att {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


FOMAML+Focal+Att 1-shot: 0.2501 +/- 0.1650  (vs FOMAML: +0.2501)
FOMAML+Focal+Att 3-shot: 0.2889 +/- 0.2410  (vs FOMAML: +0.2889)
FOMAML+Focal+Att 5-shot: 0.3171 +/- 0.1913  (vs FOMAML: +0.3171)


---
# Section 14 — FOMAML + Aug + Attention
> Combination: Augmentation + Attention U-Net.


In [ ]:
fomaml_aug_att = AttentionUNet().to(DEVICE)
fomaml_aug_att_losses, _ = fomaml_train(
    fomaml_aug_att, train_tasks, val_tasks=val_tasks, use_augmentation=True)
torch.save(fomaml_aug_att.state_dict(), 'fomaml_aug_att.pth')
losses_dict['FOMAML+Aug+Att'] = fomaml_aug_att_losses
print('FOMAML+Aug+Att saved.')


FOMAML:   4%|██▋                                                                 | 200/5000 [05:52<19:42:22, 14.78s/it]

Step  200 | loss: 1.1227 | val: 0.3365


FOMAML:   8%|█████▌                                                               | 400/5000 [10:16<6:22:30,  4.99s/it]

Step  400 | loss: 0.6968 | val: 0.4170


FOMAML:  12%|████████▎                                                            | 600/5000 [14:39<6:11:17,  5.06s/it]

...

FOMAML:  16%|███████████                                                          | 800/5000 [19:11<5:58:45,  5.13s/it]

Step  800 | loss: 0.8071 | val: 0.2806


FOMAML:  20%|█████████████▌                                                      | 1000/5000 [23:46<6:14:52,  5.62s/it]

Step 1000 | loss: 0.7265 | val: 0.2515


FOMAML:  22%|██████████████▋                                                     | 1080/5000 [25:31<1:32:20,  1.41s/it]

In [ ]:
fomaml_aug_att = AttentionUNet().to(DEVICE)
fomaml_aug_att.load_state_dict(torch.load('fomaml_aug_att.pth', map_location=DEVICE))
results['FOMAML+Aug+Att'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_aug_att, tp, n, use_augmentation=True)
         for tp in test_tasks]
    results['FOMAML+Aug+Att'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+Aug+Att {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


---
# Section 15 — FOMAML + All Improvements
> Focal Loss + Augmentation + Attention U-Net combined.


In [ ]:
fomaml_all = AttentionUNet().to(DEVICE)
fomaml_all_losses, _ = fomaml_train(
    fomaml_all, train_tasks, val_tasks=val_tasks,
    use_augmentation=True, loss_fn=focal_dice_loss)
torch.save(fomaml_all.state_dict(), 'fomaml_all.pth')
losses_dict['FOMAML+All'] = fomaml_all_losses
print('FOMAML+All saved.')


In [ ]:
fomaml_all = AttentionUNet().to(DEVICE)
fomaml_all.load_state_dict(torch.load('fomaml_all.pth', map_location=DEVICE))
results['FOMAML+All'] = {}
for n in N_SHOT_LIST:
    d = [evaluate_on_task(fomaml_all, tp, n,
                           use_augmentation=True, loss_fn=focal_dice_loss)
         for tp in test_tasks]
    results['FOMAML+All'][n] = d
    base = np.mean(results['FOMAML'][n]) if 'FOMAML' in results else 0
    print(f'FOMAML+All {n}-shot: {np.mean(d):.4f} +/- {np.std(d):.4f}  (vs FOMAML: {np.mean(d)-base:+.4f})')


# Reload all models if kernel was restarted

In [ ]:
# Reload all models if kernel was restarted
fomaml_model = UNet().to(DEVICE)
fomaml_model.load_state_dict(torch.load('fomaml.pth', map_location=DEVICE))
fomaml_focal = UNet().to(DEVICE)
fomaml_focal.load_state_dict(torch.load('fomaml_focal.pth', map_location=DEVICE))
fomaml_aug = UNet().to(DEVICE)
fomaml_aug.load_state_dict(torch.load('fomaml_aug.pth', map_location=DEVICE))
fomaml_att = AttentionUNet().to(DEVICE)
fomaml_att.load_state_dict(torch.load('fomaml_att.pth', map_location=DEVICE))
fomaml_focal_aug = UNet().to(DEVICE)
fomaml_focal_aug.load_state_dict(torch.load('fomaml_focal_aug.pth', map_location=DEVICE))
fomaml_focal_att = AttentionUNet().to(DEVICE)
fomaml_focal_att.load_state_dict(torch.load('fomaml_focal_att.pth', map_location=DEVICE))
fomaml_aug_att = AttentionUNet().to(DEVICE)
fomaml_aug_att.load_state_dict(torch.load('fomaml_aug_att.pth', map_location=DEVICE))
fomaml_all = AttentionUNet().to(DEVICE)
fomaml_all.load_state_dict(torch.load('fomaml_all.pth', map_location=DEVICE))
print('All models reloaded!')

---
# Section 16 — Full Evaluation, Plots & Save
> Run after all sections complete.


In [ ]:
# ── Full Results Table ───────────────────────────────────────────────────────
sections = [
    ('BASELINE METHODS', [
        ('Reptile',         'Reptile'),
        ('FOMAML',          'FOMAML'),
        ('FullMAML',        'Full MAML'),
        ('MetaSGD',         'Meta-SGD'),
        ('Transductive',    'Transductive'),
        ('iMAML',           'iMAML'),
        ('WeightedReptile', 'Weighted Reptile'),
        ('Baseline',        'UNet Baseline'),
    ]),
    ('SINGLE IMPROVEMENTS (on FOMAML)', [
        ('FOMAML+Focal', 'FOMAML + Focal'),
        ('FOMAML+Aug',   'FOMAML + Aug'),
        ('FOMAML+Att',   'FOMAML + Attention'),
    ]),
    ('COMBINATIONS', [
        ('FOMAML+Focal+Aug', 'FOMAML + Focal + Aug'),
        ('FOMAML+Focal+Att', 'FOMAML + Focal + Att'),
        ('FOMAML+Aug+Att',   'FOMAML + Aug + Att'),
        ('FOMAML+All',       'FOMAML + All [BEST]'),
    ]),
]

print('='*70)
print(f"{'Method':<26} {'1-shot':>10} {'3-shot':>10} {'5-shot':>10}")
print('='*70)
for section_name, methods in sections:
    print(f'\n  [{section_name}]')
    for key, name in methods:
        if key not in results: continue
        means = [f"{np.mean(results[key][n]):.3f}" for n in N_SHOT_LIST]
        print(f'  {name:<26} {means[0]:>10} {means[1]:>10} {means[2]:>10}')
print('='*70)


In [ ]:
# ── Training Curves ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
styles = {
    'Reptile':         ('steelblue',      '-'),
    'FOMAML':          ('mediumseagreen', '-'),
    'FullMAML':        ('mediumpurple',   '-'),
    'MetaSGD':         ('crimson',        '-'),
    'iMAML':           ('darkcyan',       '-'),
    'WeightedReptile': ('goldenrod',      '--'),
    'FOMAML+Focal':    ('mediumseagreen', '--'),
    'FOMAML+Aug':      ('mediumseagreen', ':'),
    'FOMAML+Att':      ('mediumseagreen', '-.'),
    'FOMAML+All':      ('black',          '-'),
}
for key, (colour, ls) in styles.items():
    if key not in losses_dict: continue
    L = losses_dict[key]
    w = max(1, len(L)//100)
    s = np.convolve(L, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(L)), s, label=key, color=colour, ls=ls, lw=1.5)
ax.set_xlabel('Outer step'); ax.set_ylabel('Loss')
ax.set_title('Meta-Training Loss — All Methods')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Methods Comparison Bar Chart ─────────────────────────────────────────────
compare_methods = [
    ('Reptile',         'Reptile',          'steelblue'),
    ('FOMAML',          'FOMAML',           'mediumseagreen'),
    ('FullMAML',        'Full MAML',        'mediumpurple'),
    ('MetaSGD',         'Meta-SGD',         'crimson'),
    ('Transductive',    'Transductive',     'goldenrod'),
    ('iMAML',           'iMAML',            'darkcyan'),
    ('WeightedReptile', 'Wtd Reptile',      'teal'),
    ('Baseline',        'UNet Baseline',    'darkorange'),
    ('FOMAML+All',      'FOMAML+All [best]','black'),
]
available = [(k,l,c) for k,l,c in compare_methods if k in results]
x     = np.arange(len(N_SHOT_LIST))
width = 0.8 / len(available)
fig, ax = plt.subplots(figsize=(16, 6))
for i, (key, label, colour) in enumerate(available):
    means  = [np.mean(results[key][n]) for n in N_SHOT_LIST]
    stds   = [np.std(results[key][n])  for n in N_SHOT_LIST]
    offset = (i - len(available)/2 + 0.5) * width
    ec     = 'gold' if key == 'FOMAML+All' else colour
    lw     = 2 if key == 'FOMAML+All' else 0
    ax.bar(x+offset, means, width, yerr=stds, label=label,
           color=colour, capsize=3, alpha=0.85, edgecolor=ec, linewidth=lw)
ax.set_xlabel('N-Shot'); ax.set_ylabel('Dice Score')
ax.set_title('All Methods Comparison')
ax.set_xticks(x); ax.set_xticklabels([f'{n}-shot' for n in N_SHOT_LIST])
ax.legend(fontsize=8, ncol=3); ax.set_ylim(0,1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Ablation Study — FOMAML Improvements ─────────────────────────────────────
ablation_groups = [
    ('FOMAML',           'FOMAML\nBaseline', 'mediumseagreen'),
    ('FOMAML+Focal',     '+Focal',            'crimson'),
    ('FOMAML+Aug',       '+Aug',              'steelblue'),
    ('FOMAML+Att',       '+Attention',        'mediumpurple'),
    ('FOMAML+Focal+Aug', '+Focal\n+Aug',     'darkorange'),
    ('FOMAML+Focal+Att', '+Focal\n+Att',     'teal'),
    ('FOMAML+Aug+Att',   '+Aug\n+Att',       'goldenrod'),
    ('FOMAML+All',       '+All\n[BEST]',     'black'),
]
available_ab = [(k,l,c) for k,l,c in ablation_groups if k in results]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax_i, n_shot in enumerate(N_SHOT_LIST):
    means = [np.mean(results[k][n_shot]) for k,l,c in available_ab]
    stds  = [np.std(results[k][n_shot])  for k,l,c in available_ab]
    cols  = [c for k,l,c in available_ab]
    bars  = axes[ax_i].bar(range(len(available_ab)), means, yerr=stds,
                            color=cols, capsize=5, alpha=0.85)
    for bar, m in zip(bars, means):
        axes[ax_i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                         f'{m:.3f}', ha='center', va='bottom', fontsize=7)
    axes[ax_i].set_xticks(range(len(available_ab)))
    axes[ax_i].set_xticklabels([l for k,l,c in available_ab], fontsize=7)
    axes[ax_i].set_ylabel('Dice Score'); axes[ax_i].set_ylim(0, 0.85)
    axes[ax_i].set_title(f'{n_shot}-shot Ablation')
    axes[ax_i].axhline(means[0], color='mediumseagreen', ls='--', alpha=0.4)
    axes[ax_i].grid(axis='y', alpha=0.3)
plt.suptitle('Ablation Study: FOMAML Improvements', fontsize=13)
plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-Task Analysis ─────────────────────────────────────────────────────────
task_names = [os.path.basename(tp) for tp in test_tasks]
key_methods = [
    ('FOMAML',       'FOMAML',           'mediumseagreen'),
    ('Transductive', 'Transductive',     'goldenrod'),
    ('FOMAML+Focal', 'FOMAML+Focal',     'crimson'),
    ('FOMAML+All',   'FOMAML+All [best]','black'),
    ('Baseline',     'UNet Baseline',    'darkorange'),
]
available_pt = [(k,l,c) for k,l,c in key_methods if k in results]
x     = np.arange(len(task_names))
width = 0.8 / len(available_pt)
fig, ax = plt.subplots(figsize=(10, 5))
for i, (key, label, colour) in enumerate(available_pt):
    dices  = results[key][5]
    offset = (i - len(available_pt)/2 + 0.5) * width
    ax.bar(x+offset, dices, width, label=label, color=colour, alpha=0.85)
ax.set_xlabel('Task'); ax.set_ylabel('Dice Score')
ax.set_title('Per-Task Performance (5-shot)')
ax.set_xticks(x); ax.set_xticklabels(task_names)
ax.legend(fontsize=9); ax.set_ylim(0,1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_task.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Qualitative Visualisation ─────────────────────────────────────────────────
def visualise_all(task_path, n_shot=5, n_examples=3):
    _g = globals()

    # Registry: (variable_name, display_label, use_aug, loss_fn)
    _registry = [
        ('fomaml_model',     'FOMAML\nBaseline',  False, bce_dice_loss),
        ('fomaml_focal',     'FOMAML\n+Focal',    False, focal_dice_loss),
        ('fomaml_aug',       'FOMAML\n+Aug',      True,  bce_dice_loss),
        ('fomaml_att',       'FOMAML\n+Attention',False, bce_dice_loss),
        ('fomaml_focal_aug', '+Focal\n+Aug',      True,  focal_dice_loss),
        ('fomaml_focal_att', '+Focal\n+Att',      False, focal_dice_loss),
        ('fomaml_aug_att',   '+Aug\n+Att',        True,  bce_dice_loss),
        ('fomaml_all',       'FOMAML\n+All',      True,  focal_dice_loss),
    ]

    adapted_list = []

    # Standard models
    for var_name, label, use_aug, loss_fn in _registry:
        if var_name in _g:
            adapted = few_shot_adapt(
                _g[var_name], task_path, n_shot,
                use_augmentation=use_aug, loss_fn=loss_fn)
            adapted_list.append((label, adapted))

    # Transductive inserted after FOMAML baseline
    if 'fomaml_model' in _g:
        trans = transductive_adapt(
            _g['fomaml_model'], task_path, n_shot, loss_fn=bce_dice_loss)
        adapted_list.insert(1, ('Transductive\nBase', trans))

    if not adapted_list:
        print('No models loaded — run sections first.')
        return

    dataset = TaskDataset(task_path)
    indices = random.sample(range(len(dataset)), k=min(n_examples, len(dataset)))
    ncols   = 2 + len(adapted_list)
    cols    = ['Input', 'Ground\nTruth'] + [label for label, _ in adapted_list]

    fig, axes = plt.subplots(n_examples, ncols,
                              figsize=(3.2*ncols, 3.5*n_examples))
    if n_examples == 1: axes = axes[np.newaxis, :]
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=7, fontweight='bold')

    for row, idx in enumerate(indices):
        img, mask = dataset[idx]
        img_b = img.unsqueeze(0).to(DEVICE)
        preds = [img.squeeze(), mask.squeeze()]
        with torch.no_grad():
            for _, adapted in adapted_list:
                pred = (torch.sigmoid(adapted(img_b)) > 0.5).float().squeeze().cpu()
                preds.append(pred)
        for ax, data in zip(axes[row], preds):
            ax.imshow(data, cmap='gray'); ax.axis('off')

    task_name = os.path.basename(task_path)
    fig.suptitle(f'{task_name} — {n_shot}-shot predictions', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'qualitative_{task_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

for task in test_tasks:
    visualise_all(task, n_shot=5)


In [ ]:
# ── Save Everything ───────────────────────────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir  = f'results_{timestamp}'
os.makedirs(save_dir, exist_ok=True)

# Models
for fname in ['reptile.pth','fomaml.pth','maml.pth','metasgd.pth',
              'imaml.pth','wreptile.pth',
              'fomaml_focal.pth','fomaml_aug.pth','fomaml_att.pth',
              'fomaml_focal_aug.pth','fomaml_focal_att.pth',
              'fomaml_aug_att.pth','fomaml_all.pth']:
    if os.path.exists(fname): shutil.copy(fname, os.path.join(save_dir, fname))

# Results JSON
results_serial = {
    method: {str(n): [float(d) for d in dices] for n, dices in shots.items()}
    for method, shots in results.items()
}
with open(os.path.join(save_dir,'results.json'),'w') as f:
    json.dump(results_serial, f, indent=2)

# CSV summary
with open(os.path.join(save_dir,'results_summary.csv'),'w',newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Method','N-Shot','Mean Dice','Std']+
                    [os.path.basename(t) for t in test_tasks])
    for sec_name, methods in sections:
        for key, name in methods:
            if key not in results: continue
            for n in N_SHOT_LIST:
                d = results[key][n]
                writer.writerow([name,f'{n}-shot',
                                  f'{np.mean(d):.4f}',f'{np.std(d):.4f}']+
                                [f'{x:.4f}' for x in d])

# Training losses
with open(os.path.join(save_dir,'training_losses.pkl'),'wb') as f:
    pickle.dump(losses_dict, f)

# Figures
for fig_name in ['training_curves.png','methods_comparison.png',
                  'ablation_study.png','per_task.png']:
    if os.path.exists(fig_name): shutil.copy(fig_name, os.path.join(save_dir, fig_name))
for task in test_tasks:
    fn = f'qualitative_{os.path.basename(task)}.png'
    if os.path.exists(fn): shutil.copy(fn, os.path.join(save_dir, fn))

print(f'Saved to: {save_dir}/')
for fname in sorted(os.listdir(save_dir)):
    sz = os.path.getsize(os.path.join(save_dir,fname))
    print(f'  {fname:<50} {sz/1024:.1f} KB')
